In [1]:
!pip -q install python-dotenv reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.1 MB/s eta 0:00:00


In [2]:
import os

PROJECT_DIR = "/content/email-sms-automator"

folders = [
    "src",
    "data",
    "data/attachments"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_DIR, folder), exist_ok=True)

print("สร้างโฟลเดอร์โปรเจกต์เรียบร้อยแล้ว")
for folder in folders:
    print("-", folder)

สร้างโฟลเดอร์โปรเจกต์เรียบร้อยแล้ว
- src
- data
- data/attachments


In [3]:
%%writefile /content/email-sms-automator/src/__init__.py

# This file marks src as a Python package.

Writing /content/email-sms-automator/src/__init__.py


In [4]:
%%writefile /content/email-sms-automator/src/utils.py

import logging
import os
from dotenv import load_dotenv

# โหลดค่าจาก .env ถ้ามีไฟล์นี้
load_dotenv(
    dotenv_path=os.path.join(
        os.path.dirname(os.path.dirname(__file__)),
        ".env"
    )
)

def setup_logging(name, level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)

    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger

def get_env_variable(var_name):
    value = os.getenv(var_name)

    if not value:
        raise ValueError(
            f"ยังไม่ได้กำหนดตัวแปร {var_name} กรุณาตรวจสอบการตั้งค่า"
        )

    return value

def ensure_directory_exists(path):
    os.makedirs(path, exist_ok=True)
    return path

Writing /content/email-sms-automator/src/utils.py


In [5]:
%%writefile /content/email-sms-automator/src/email_sender.py

import os
import smtplib
from email.message import EmailMessage
from .utils import setup_logging

logger = setup_logging(__name__)


class EmailSender:
    def __init__(self, smtp_server, smtp_port, sender_email, sender_password):
        self.smtp_server = smtp_server
        self.smtp_port = int(smtp_port)
        self.sender_email = sender_email
        self.sender_password = sender_password

    def send_email(
        self,
        recipient_email,
        subject,
        body,
        attachment_path=None,
        is_html=False
    ):
        message = EmailMessage()
        message["From"] = self.sender_email
        message["To"] = recipient_email
        message["Subject"] = subject

        if is_html:
            message.set_content("กรุณาเปิดอีเมลด้วยโปรแกรมที่รองรับ HTML")
            message.add_alternative(body, subtype="html")
        else:
            message.set_content(body)

        if attachment_path:
            if not os.path.isfile(attachment_path):
                logger.error("ไม่พบไฟล์แนบ: %s", attachment_path)
                return False

            with open(attachment_path, "rb") as file:
                file_data = file.read()

            message.add_attachment(
                file_data,
                maintype="application",
                subtype="pdf",
                filename=os.path.basename(attachment_path)
            )

        try:
            with smtplib.SMTP(
                self.smtp_server, self.smtp_port, timeout=30
            ) as server:
                server.ehlo()
                server.starttls()
                server.ehlo()
                server.login(self.sender_email, self.sender_password)
                server.send_message(message)

            logger.info("ส่งอีเมลสำเร็จไปยัง %s", recipient_email)
            return True

        except (smtplib.SMTPException, OSError) as error:
            logger.error(
                "ส่งอีเมลไม่สำเร็จ (%s): %s",
                type(error).__name__,
                error
            )
            return False

Writing /content/email-sms-automator/src/email_sender.py


In [6]:
%%writefile /content/email-sms-automator/src/sms_gateway.py

from .email_sender import EmailSender
from .utils import setup_logging

logger = setup_logging(__name__)


class SMSGateway:
    def __init__(self, email_sender):
        if not isinstance(email_sender, EmailSender):
            raise TypeError("ต้องส่ง EmailSender เข้ามา")

        self.email_sender = email_sender

    def send_sms(self, gateway_address, message):
        if "@" not in gateway_address:
            logger.error("ต้องระบุอีเมล Gateway ที่ถูกต้อง")
            return False

        logger.info("กำลังส่งข้อความผ่าน Email-to-SMS Gateway")

        return self.email_sender.send_email(
            recipient_email=gateway_address,
            subject="",
            body=message
        )

Writing /content/email-sms-automator/src/sms_gateway.py


In [7]:
%%writefile /content/email-sms-automator/src/email_receiver.py

import imaplib
import email
import os
from email.header import decode_header
from .utils import setup_logging, ensure_directory_exists

logger = setup_logging(__name__)


def decode_mime_header(value):
    if not value:
        return ""

    decoded = []
    for part, charset in decode_header(value):
        if isinstance(part, bytes):
            decoded.append(
                part.decode(charset or "utf-8", errors="replace")
            )
        else:
            decoded.append(part)

    return "".join(decoded)


class EmailReceiver:
    def __init__(self, imap_server, sender_email, sender_password):
        self.imap_server = imap_server
        self.sender_email = sender_email
        self.sender_password = sender_password

    def search_emails(self, criteria="UNSEEN", from_sender=None,
                      subject_contains=None):
        mail = None

        try:
            mail = imaplib.IMAP4_SSL(self.imap_server)
            mail.login(self.sender_email, self.sender_password)
            mail.select("INBOX")

            # สร้างเงื่อนไข IMAP search
            query = [criteria]

            if from_sender:
                query.extend(["FROM", from_sender])

            if subject_contains:
                query.extend(["SUBJECT", subject_contains])

            status, data = mail.search(None, *query)

            if status != "OK":
                logger.error("ค้นหาอีเมลไม่สำเร็จ")
                return []

            message_ids = data[0].split()
            logger.info("พบอีเมล %d รายการ", len(message_ids))
            return message_ids

        except (imaplib.IMAP4.error, OSError) as error:
            logger.error(
                "เชื่อมต่อหรือค้นหา IMAP ไม่สำเร็จ: %s", error
            )
            return []

        finally:
            if mail is not None:
                try:
                    mail.logout()
                except Exception:
                    pass

    def fetch_email_content(self, message_id, mark_as_read=True,
                            download_attachments_dir=None):
        mail = None

        try:
            mail = imaplib.IMAP4_SSL(self.imap_server)
            mail.login(self.sender_email, self.sender_password)
            mail.select("INBOX")

            status, data = mail.fetch(message_id, "(RFC822)")
            if status != "OK" or not data or not isinstance(data[0], tuple):
                logger.error("อ่านอีเมลไม่สำเร็จ")
                return None

            msg = email.message_from_bytes(data[0][1])
            body_text = ""
            downloaded = []

            if download_attachments_dir:
                ensure_directory_exists(download_attachments_dir)

            for part in msg.walk():
                if part.is_multipart():
                    continue

                filename = part.get_filename()
                disposition = part.get_content_disposition()

                if filename or disposition == "attachment":
                    if filename and download_attachments_dir:
                        # ป้องกันชื่อไฟล์แนบมี path แอบแฝง
                        safe_name = os.path.basename(
                            decode_mime_header(filename)
                        )
                        if safe_name:
                            path = os.path.join(
                                download_attachments_dir, safe_name
                            )
                            payload = part.get_payload(decode=True) or b""

                            with open(path, "wb") as file:
                                file.write(payload)

                            downloaded.append(path)
                    continue

                if part.get_content_type() == "text/plain":
                    payload = part.get_payload(decode=True)
                    if payload:
                        charset = part.get_content_charset() or "utf-8"
                        body_text += payload.decode(
                            charset, errors="replace"
                        )

            if mark_as_read:
                mail.store(message_id, "+FLAGS", "\\Seen")

            return {
                "sender": decode_mime_header(msg.get("From")),
                "subject": decode_mime_header(msg.get("Subject")),
                "body_snippet": body_text[:200],
                "attachments_downloaded": downloaded
            }

        except (imaplib.IMAP4.error, OSError, ValueError) as error:
            logger.error("อ่านหรือแยกข้อมูลอีเมลไม่สำเร็จ: %s", error)
            return None

        finally:
            if mail is not None:
                try:
                    mail.logout()
                except Exception:
                    pass

Writing /content/email-sms-automator/src/email_receiver.py


In [8]:
from reportlab.pdfgen import canvas

pdf_path = os.path.join(
    PROJECT_DIR, "data", "dummy_report.pdf"
)

pdf = canvas.Canvas(pdf_path)
pdf.setTitle("Dummy Report")
pdf.setFont("Helvetica-Bold", 18)
pdf.drawString(72, 760, "Automated Report")
pdf.setFont("Helvetica", 12)
pdf.drawString(72, 720, "This is a sample PDF attachment.")
pdf.drawString(72, 695, "Created for LabX BASIC Week 12.")
pdf.save()

print("สร้างไฟล์แนบ:", pdf_path)

สร้างไฟล์แนบ: /content/email-sms-automator/data/dummy_report.pdf


In [9]:
env_path = os.path.join(PROJECT_DIR, ".env")

env_template = """SENDER_EMAIL=
SENDER_APP_PASSWORD=
SMTP_SERVER=smtp.gmail.com
SMTP_PORT=587
IMAP_SERVER=imap.gmail.com
TEST_RECIPIENT_EMAIL=
TEST_SMS_GATEWAY_ADDRESS=
SEND_REAL_EMAILS=false
"""

with open(env_path, "w", encoding="utf-8") as file:
    file.write(env_template)

gitignore_path = os.path.join(PROJECT_DIR, ".gitignore")

gitignore_content = """# Secrets
.env

# Python cache
__pycache__/
*.pyc

# Downloaded attachments
data/attachments/

# Logs
*.log
"""

with open(gitignore_path, "w", encoding="utf-8") as file:
    file.write(gitignore_content)

print("สร้าง .env และ .gitignore แล้ว")
print("ไฟล์ .env ยังไม่มีข้อมูลบัญชีจริง")

สร้าง .env และ .gitignore แล้ว
ไฟล์ .env ยังไม่มีข้อมูลบัญชีจริง


In [16]:
from getpass import getpass

sender_email = input("Gmail ผู้ส่ง: ").strip()
app_password = getpass("Gmail App Password (ซ่อนข้อความ): ").strip()
recipient_email = input("อีเมลผู้รับสำหรับทดสอบ: ").strip()
sms_gateway = input(
    "Email-to-SMS Gateway (เว้นว่างได้): "
).strip()

if sender_email and app_password and recipient_email:
    env_content = f"""SENDER_EMAIL={sender_email}
SENDER_APP_PASSWORD={app_password}
SMTP_SERVER=smtp.gmail.com
SMTP_PORT=587
IMAP_SERVER=imap.gmail.com
TEST_RECIPIENT_EMAIL={recipient_email}
TEST_SMS_GATEWAY_ADDRESS={sms_gateway}
SEND_REAL_EMAILS=false
"""

    with open(env_path, "w", encoding="utf-8") as file:
        file.write(env_content)

    print("บันทึกการตั้งค่าแล้ว โดยยังปิดการส่งจริงไว้")
else:
    print("ยังไม่ได้บันทึก กรุณากรอกอีเมลและ App Password ให้ครบ")

Gmail ผู้ส่ง: Chaiwat Jamklang
Gmail App Password (ซ่อนข้อความ): ··········
อีเมลผู้รับสำหรับทดสอบ: chaiwat.jam@kkumail.com
Email-to-SMS Gateway (เว้นว่างได้): 
บันทึกการตั้งค่าแล้ว โดยยังปิดการส่งจริงไว้


In [11]:
%%writefile /content/email-sms-automator/main.py

import os
import time
from dotenv import load_dotenv

from src.utils import (
    setup_logging,
    get_env_variable,
    ensure_directory_exists
)
from src.email_sender import EmailSender
from src.email_receiver import EmailReceiver
from src.sms_gateway import SMSGateway

logger = setup_logging("main")


def main():
    base_dir = os.path.dirname(os.path.abspath(__file__))
    load_dotenv(os.path.join(base_dir, ".env"), override=True)

    data_dir = os.path.join(base_dir, "data")
    attachments_dir = os.path.join(data_dir, "attachments")
    ensure_directory_exists(attachments_dir)

    attachment_path = os.path.join(data_dir, "dummy_report.pdf")

    if not os.path.exists(attachment_path):
        logger.error("ไม่พบไฟล์แนบ dummy_report.pdf")
        return

    # ตรวจสอบโหมดก่อนเชื่อมต่อเครือข่าย
    send_real = os.getenv("SEND_REAL_EMAILS", "false").lower() == "true"

    if not send_real:
        logger.info("โหมดจำลอง: ไม่มีการส่งอีเมลหรือ SMS จริง")
        logger.info("รายการที่ระบบเตรียมทำ:")
        logger.info("1. ส่งอีเมลข้อความธรรมดา")
        logger.info("2. ส่งอีเมลพร้อมไฟล์แนบ PDF")
        logger.info("3. ส่ง SMS ผ่าน Email Gateway หากกำหนด Gateway")
        logger.info("4. ค้นหาอีเมลที่ยังไม่อ่านและดาวน์โหลดไฟล์แนบ")
        logger.info(
            "หากต้องการทดสอบจริง ให้ตั้ง SEND_REAL_EMAILS=true ใน .env"
        )
        return

    try:
        sender_email = get_env_variable("SENDER_EMAIL")
        sender_password = get_env_variable("SENDER_APP_PASSWORD")
        smtp_server = get_env_variable("SMTP_SERVER")
        smtp_port = int(get_env_variable("SMTP_PORT"))
        imap_server = get_env_variable("IMAP_SERVER")
        recipient_email = get_env_variable("TEST_RECIPIENT_EMAIL")

    except (ValueError, TypeError) as error:
        logger.error("การตั้งค่าไม่ครบ: %s", error)
        return

    email_sender = EmailSender(
        smtp_server, smtp_port, sender_email, sender_password
    )
    email_receiver = EmailReceiver(
        imap_server, sender_email, sender_password
    )
    sms_gateway = SMSGateway(email_sender)

    # Task 1: ส่งอีเมลข้อความธรรมดา
    logger.info("Task 1: ส่งอีเมลข้อความธรรมดา")
    email_sender.send_email(
        recipient_email,
        "LabX BASIC Week 12 - Test Email",
        "Hello! This is an automated test email from Python."
    )

    # Task 2: ส่งอีเมลพร้อมไฟล์แนบ
    logger.info("Task 2: ส่งอีเมลพร้อม PDF")
    email_sender.send_email(
        recipient_email,
        "LabX BASIC Week 12 - PDF Attachment",
        "Please find the sample report attached.",
        attachment_path=attachment_path
    )

    # Task 3: ส่ง SMS ผ่าน Gateway เฉพาะเมื่อมีการกำหนด
    gateway_address = os.getenv("TEST_SMS_GATEWAY_ADDRESS", "").strip()

    if gateway_address:
        logger.info("Task 3: ส่งข้อความผ่าน Email-to-SMS Gateway")
        sms_gateway.send_sms(
            gateway_address,
            "LabX BASIC Week 12: Automated notification test."
        )
    else:
        logger.info("ข้าม SMS: ยังไม่ได้กำหนด TEST_SMS_GATEWAY_ADDRESS")

    # Task 4: ส่งอีเมลสาธิตเข้าบัญชีตัวเอง
    demo_subject = "Python Week12 Receive Demo"

    logger.info("Task 4: ส่งอีเมลสำหรับทดสอบการรับ")
    sent = email_sender.send_email(
        sender_email,
        demo_subject,
        "This message is for the IMAP receiving demonstration.",
        attachment_path=attachment_path
    )

    if sent:
        logger.info("รอระบบอีเมลรับข้อความสักครู่")
        time.sleep(8)

        uids = email_receiver.search_emails(
            criteria="UNSEEN",
            from_sender=sender_email,
            subject_contains=demo_subject
        )

        for uid in uids:
            details = email_receiver.fetch_email_content(
                uid,
                mark_as_read=True,
                download_attachments_dir=attachments_dir
            )

            if details:
                logger.info("Sender: %s", details["sender"])
                logger.info("Subject: %s", details["subject"])
                logger.info("Body: %s", details["body_snippet"])
                logger.info(
                    "Downloaded attachments: %s",
                    details["attachments_downloaded"]
                )

    logger.info("งานสาธิตเสร็จสิ้น")


if __name__ == "__main__":
    main()

Writing /content/email-sms-automator/main.py


In [17]:
%cd /content/email-sms-automator
!python main.py

/content/email-sms-automator
2026-10-13 09:31:25,643 - main - INFO - โหมดจำลอง: ไม่มีการส่งอีเมลหรือ SMS จริง
2026-10-13 09:31:25,643 - main - INFO - รายการที่ระบบเตรียมทำ:
2026-10-13 09:31:25,643 - main - INFO - 1. ส่งอีเมลข้อความธรรมดา
2026-10-13 09:31:25,643 - main - INFO - 2. ส่งอีเมลพร้อมไฟล์แนบ PDF
2026-10-13 09:31:25,643 - main - INFO - 3. ส่ง SMS ผ่าน Email Gateway หากกำหนด Gateway
2026-10-13 09:31:25,643 - main - INFO - 4. ค้นหาอีเมลที่ยังไม่อ่านและดาวน์โหลดไฟล์แนบ
2026-10-13 09:31:25,643 - main - INFO - หากต้องการทดสอบจริง ให้ตั้ง SEND_REAL_EMAILS=true ใน .env


In [13]:
import os

required_files = [
    "src/__init__.py",
    "src/utils.py",
    "src/email_sender.py",
    "src/email_receiver.py",
    "src/sms_gateway.py",
    "data/dummy_report.pdf",
    ".env",
    ".gitignore",
    "main.py"
]

print("===== Project File Check =====")

for filename in required_files:
    path = os.path.join(PROJECT_DIR, filename)

    if os.path.isfile(path):
        print(f"OK: {filename}")
    else:
        print(f"MISSING: {filename}")

===== Project File Check =====
OK: src/__init__.py
OK: src/utils.py
OK: src/email_sender.py
OK: src/email_receiver.py
OK: src/sms_gateway.py
OK: data/dummy_report.pdf
OK: .env
OK: .gitignore
OK: main.py


In [14]:
%cd /content/email-sms-automator

from src.utils import setup_logging, ensure_directory_exists
from src.email_sender import EmailSender
from src.email_receiver import EmailReceiver
from src.sms_gateway import SMSGateway

print("Import test: PASSED")
print("EmailSender: OK")
print("EmailReceiver: OK")
print("SMSGateway: OK")

/content/email-sms-automator
Import test: PASSED
EmailSender: OK
EmailReceiver: OK
SMSGateway: OK


In [15]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/email-sms-automator",
    "zip",
    root_dir="/content",
    base_dir="email-sms-automator"
)

print("สร้าง ZIP สำเร็จ:", zip_path)
files.download(zip_path)

สร้าง ZIP สำเร็จ: /content/email-sms-automator.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>